----------------------------------------------------------------------------------------------------
**PRICING IMPACT SIMULATION WITH AN A/B TEST**

Let's procede with the A/B testing, even though we don't have real data regarding the acceptance or not of the personalized prices. For this reason, we are going to work with an offline post-hoc simulation. It means that we don't do a real A/B live test, but we do synthetize two groups A and B starting from our dataset.

We need to do:

1) Divide the dataset in two groups A and B
2) Assign different prices to the groups, for example the original prices to the A group and the new ones to the B group
3) Simulate the probability of conversion as an inverse function of the price
4) Calculate the metrics: conversion rate and net revenue
5) Statistical analysis to better understand if the difference between A and B is significative.

In [1]:
#Let's firstly import the elaborated version of the original dataset, from main project
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

dp = pd.read_csv('dynamic_pricing_elaborated.csv')

In [2]:
#1. Creation of the two virtual groups
np.random.seed(42)
dp['group'] = np.random.choice(['A', 'B'], size=len(dp))

In [3]:
#2. Assignment of prices to A and B
dp['simulated_price'] = dp['Historical_Cost_of_Ride']
dp.loc[dp['group'] == 'B', 'simulated_price'] = dp['Calculated_Fare']

Group A was associated with the original prices, whereas group B was assigned the new prices computed in the earlier code.

In [5]:
#3. Simulation of the probability
from scipy.special import expit  # sigmoid function

#Build up a conversion probability inversely proportional to the price
dp['conversion_prob'] = expit(-0.5 * (dp['simulated_price'] - dp['simulated_price'].mean()))

# Simulate the conversion (1 = rider has accepted the price)
dp['conversion'] = np.random.binomial(1, dp['conversion_prob'])

In [6]:
#4. Calculation of metrics

# Price - simulated costs (ex. 70% of price as a cost)
dp['cost'] = dp['simulated_price'] * 0.7
dp['net_revenue'] = dp['conversion'] * (dp['simulated_price'] - dp['cost'])

# Conversion rate and net revenue per group
results = dp.groupby('group').agg({
    'conversion': 'mean',
    'net_revenue': 'mean',
    'simulated_price': 'mean'
}).rename(columns={
    'conversion': 'conversion_rate',
    'net_revenue': 'avg_net_revenue',
    'simulated_price': 'avg_price'
})

print(results)

       conversion_rate  avg_net_revenue   avg_price
group                                              
A             0.757143        65.617414  371.059028
B             0.454902        39.586554  655.954837


In the following section, a t-test will be conducted to determine whether the two groups differ in terms of their means. If the means are equal, it suggests that the modified prices had no effect on riders’ decision-making. Conversely, if the means differ, this indicates that behavior was influenced, though not necessarily in a positive way—for instance, riders might actually prefer the original prices.

In [7]:
#5. Statistical analysis
from scipy.stats import ttest_ind

a = dp[dp['group'] == 'A']['net_revenue']
b = dp[dp['group'] == 'B']['net_revenue']

t_stat, p_value = ttest_ind(a, b)
print(f"T-stat: {t_stat:.3f}, P-value: {p_value:.16f}")

T-stat: 8.111, P-value: 0.0000000000000015


From the results, we observe that although the average price of group A is lower, this group shows a considerably higher conversion rate. This means that, in the simulation, more riders in group A accepted the price compared to those in group B. Therefore, the prices associated with group A appear to better incentivize riders, allowing them to generate more net revenue despite the lower prices.

The t-test instead shows that the mean net revenues are significantly different (with a very low p-value and a very high t-statistic). Therefore, the null hypothesis that the two population means are the same can be decisively rejected.

We therefore conclude that, at least based on the simulation, the original prices appear to be better (or at least preferable). A clear next step would be to collect real data on ride acceptance rather than relying solely on a simulation. Furthermore, it could be useful to revisit the price modification process with alternative approaches, by going back to the previous code and re-running the steps using different pricing strategies.